In [18]:
import os
import json
import numpy as np
from scipy.interpolate import CubicSpline

# Relative paths based on your project structure
PROJECT_ROOT = ".."
JSON_INPUT_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "bench_press")
JSON_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "interpolated_json", "bench_press")

SUB_FOLDERS = ["Abdillah", "RepCount"]

In [19]:
import os
import json
import numpy as np
from scipy.interpolate import CubicSpline

def interpolate_skeleton(json_data):
    """
    Extracts landmarks from the 'frames' list and applies Cubic Spline.
    Handles 'null' landmarks and dictionary-style frame structures.
    """
    # 1. Access the 'frames' list
    frames_list = json_data.get('frames', [])
    if not frames_list:
        return json_data

    # 2. Extract landmarks into a list of lists
    # Replace 'null' or missing landmarks with [0,0,0,0] to maintain shape
    extracted_coords = []
    for f in frames_list:
        landmarks = f.get('landmarks')
        if landmarks is None:
            extracted_coords.append([[0.0, 0.0, 0.0, 0.0]] * 33)
        else:
            extracted_coords.append(landmarks)

    # 3. Convert to numpy array (Frames, 33, 4)
    try:
        coords = np.array(extracted_coords, dtype=np.float64)
    except Exception as e:
        print(f"⚠️ Array creation failed: {e}")
        return json_data
    
    if coords.ndim != 3:
        return json_data

    frames_idx = np.arange(len(coords))
    interpolated = np.copy(coords)

    # 4. Apply Spline to x, y, z (ignoring the 4th visibility/confidence value)
    for joint in range(33):
        for axis in range(3): # x, y, z
            values = coords[:, joint, axis]
            valid_mask = (values != 0) & (~np.isnan(values))
            
            # Bridge occlusions if we have enough data points
            if np.sum(valid_mask) > 3:
                cs = CubicSpline(frames_idx[valid_mask], values[valid_mask])
                interpolated[~valid_mask, joint, axis] = cs(frames_idx[~valid_mask])
                
    # 5. Reconstruct the original JSON structure with cleaned data
    new_frames = []
    for i, f in enumerate(frames_list):
        new_frame = f.copy()
        new_frame['landmarks'] = interpolated[i].tolist()
        new_frames.append(new_frame)
    
    output_json = json_data.copy()
    output_json['frames'] = new_frames
    return output_json

In [20]:
PROJECT_ROOT = ".."
JSON_INPUT_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "bench_press")
JSON_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "interpolated_json", "bench_press")
SUB_FOLDERS = ["Abdillah", "RepCount"]

print("🚀 Starting Deep-Path Interpolation...")

target_dirs = [JSON_INPUT_DIR] + [os.path.join(JSON_INPUT_DIR, sub) for sub in SUB_FOLDERS]

for current_in_path in target_dirs:
    if not os.path.exists(current_in_path): continue
    
    rel_path = os.path.relpath(current_in_path, JSON_INPUT_DIR)
    current_out_path = os.path.normpath(os.path.join(JSON_OUTPUT_DIR, rel_path))
    os.makedirs(current_out_path, exist_ok=True)
    
    for f_name in [f for f in os.listdir(current_in_path) if f.endswith('.json')]:
        with open(os.path.join(current_in_path, f_name), 'r') as f:
            try:
                data = json.load(f)
                clean_data = interpolate_skeleton(data)
                
                with open(os.path.join(current_out_path, f_name), 'w') as out_f:
                    json.dump(clean_data, out_f)
                print(f"  ✅ Interpolated: {os.path.join(rel_path, f_name)}")
            except Exception as e:
                print(f"  ❌ Failed {f_name}: {e}")

print("\n✨ All skeletal gaps filled. Ready for ST-GCN analysis.")

🚀 Starting Deep-Path Interpolation...
  ✅ Interpolated: ./bench press_28_aug_75_flip.json
  ✅ Interpolated: ./stu8_11_aug_65_speed.json
  ✅ Interpolated: ./bench press_53_aug_7_speed.json
  ✅ Interpolated: ./bench press_18_aug_38_speed.json
  ✅ Interpolated: ./bench press_12_aug_82_flip.json
  ✅ Interpolated: ./stu8_13_aug_50_speed.json
  ✅ Interpolated: ./stu10_4_aug_19_speed.json
  ✅ Interpolated: ./stu5_5_aug_33_speed.json
  ✅ Interpolated: ./bench press_55_aug_26_flip.json
  ✅ Interpolated: ./stu8_15_aug_24_speed.json
  ✅ Interpolated: ./stu10_12_aug_27_noise.json
  ✅ Interpolated: ./stu7_11_aug_29_speed.json
  ✅ Interpolated: ./stu1_6_aug_59_noise.json
  ✅ Interpolated: ./stu4_4_aug_64_noise.json
  ✅ Interpolated: ./stu5_10_aug_83_flip.json
  ✅ Interpolated: ./stu1_3_aug_30_flip.json
  ✅ Interpolated: ./bench press_21_aug_25_noise.json
  ✅ Interpolated: ./bench press_23_aug_67_flip.json
  ✅ Interpolated: ./stu10_7_aug_76_noise.json
  ✅ Interpolated: ./bench press_43_aug_36_flip.js